In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.inspection import permutation_importance

# ML Models
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

# Set plotting style
sns.set_theme(style="whitegrid")
plt.rcParams.update({'font.size': 10})

In [2]:
df_n=pd.read_csv(r"C:\Users\SOLOMON\Downloads\Telegram Desktop\weatherdata.csv")


In [3]:
df_n

,Date,Longitude,Latitude,Elevation,Max Temperature,Min Temperature,Wind,Relative Humidity,Solar,Precipitation
0,1/1/79,37.5,6.71292,1494,30.713,11.719,1.595418,0.418255,23.386395,0.000000
1,1/2/79,37.5,6.71292,1494,30.872,10.407,1.726672,0.432899,25.572441,0.000000
2,1/3/79,37.5,6.71292,1494,28.579,10.795,1.870545,0.456901,25.568213,0.000000
3,1/4/79,37.5,6.71292,1494,29.383,10.712,1.704374,0.463286,25.659900,0.000000
4,1/5/79,37.5,6.71292,1494,30.837,9.487,1.719667,0.394840,25.737013,0.000000
...,...,...,...,...,...,...,...,...,...,...
12982,7/27/14,37.5,6.71292,1494,21.351,15.630,1.123813,0.898494,15.266439,33.345219
12983,7/28/14,37.5,6.71292,1494,25.326,15.266,1.128937,0.840943,21.724500,11.767387
12984,7/29/14,37.5,6.71292,1494,23.138,13.402,1.297933,0.799027,18.358496,1.153566
12985,7/30/14,37.5,6.71292,1494,25.306,12.483,1.690213,0.739780,23.324918,0.418854


In [ ]:


# ---------------------------------------------------------
# 1. GENERATE SYNTHETIC LARGE WEATHER DATASET
# ---------------------------------------------------------
def generate_weather_data(n_samples=5000, random_state=42):
    np.random.seed(random_state)
    
    # Generate continuous timestamp range
    dates = pd.date_range(start="2022-01-01", periods=n_samples, freq="H")
    df = pd.DataFrame({"datetime": dates})
    
    # Generate raw weather features
    df['temperature'] = 15 + 10 * np.sin(2 * np.pi * df.index / 8760) + np.random.normal(0, 3, n_samples)
    df['humidity'] = np.clip(60 - 1.5 * df['temperature'] + np.random.normal(0, 10, n_samples), 10, 100)
    df['solar_radiation'] = np.clip(500 * np.sin(2 * np.pi * (df['datetime'].dt.hour - 6) / 24) + np.random.normal(0, 50, n_samples), 0, None)
    df['wind_speed'] = np.random.gamma(shape=2, scale=3, size=n_samples)
    df['pressure'] = 1013.25 + np.random.normal(0, 10, n_samples)
    
    # Define target variable: "target_temp_next_hour" (Weather forecasting target)
    # Target heavily depends on current temp, solar radiation, humidity, and day of year
    day_of_year = df['datetime'].dt.dayofyear
    df['target_temp_next_hour'] = (
        0.85 * df['temperature'] 
        + 0.005 * df['solar_radiation'] 
        - 0.03 * df['humidity'] 
        + 2 * np.sin(2 * np.pi * day_of_year / 365) 
        + np.random.normal(0, 1, n_samples)
    )
    return df

# Create Dataset
raw_df = generate_weather_data(n_samples=8760, random_state=42) # 1 full year hourly data

# ---------------------------------------------------------
# 2. FEATURE EXTRACTION & ENGINEERING
# ---------------------------------------------------------
def extract_weather_features(df):
    data = df.copy()
    
    # Date / Time Feature Extraction
    data['day_of_week'] = data['datetime'].dt.dayofweek
    data['day_of_year'] = data['datetime'].dt.dayofyear
    data['quarter'] = data['datetime'].dt.quarter
    data['month'] = data['datetime'].dt.month
    data['hour'] = data['datetime'].dt.hour
    
    # Cyclical Transformation for time variables (captures continuous nature)
    data['sin_hour'] = np.sin(2 * np.pi * data['hour'] / 24.0)
    data['cos_hour'] = np.cos(2 * np.pi * data['hour'] / 24.0)
    
    # Drop raw timestamp
    data = data.drop(columns=['datetime'])
    return data

df_processed = extract_weather_features(raw_df)

# Separate Features (X) and Target (y)
X = df_processed.drop(columns=['target_temp_next_hour'])
y = df_processed['target_temp_next_hour']

# Train-Test Split (80/20)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# ---------------------------------------------------------
# 3. MODEL DICTIONARY & PIPELINE EXECUTION
# ---------------------------------------------------------
ml_models = {
    "Linear Regression": LinearRegression(),
    "Ridge Regression": Ridge(alpha=1.0),
    "Lasso": Lasso(alpha=100.0, max_iter=10000, random_state=42),
    "KNN": KNeighborsRegressor(),
    "Decision Tree": DecisionTreeRegressor(random_state=42),
    "Random Forest": RandomForestRegressor(n_estimators=100, random_state=42),
    "Gradient Boosting": GradientBoostingRegressor(random_state=42),
}

# Store evaluation results
results = []
feature_importances_dict = {}

for name, model in ml_models.items():
    # Scale features inside pipeline to maintain best practice for distance/linear models
    pipeline = Pipeline([
        ('scaler', StandardScaler()),
        ('regressor', model)
    ])
    
    # Fit pipeline
    pipeline.fit(X_train, y_train)
    
    # Predict
    y_pred = pipeline.predict(X_test)
    
    # Evaluate Metrics
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    
    results.append({
        "Model": name,
        "RMSE": round(rmse, 4),
        "MAE": round(mae, 4),
        "R2 Score": round(r2, 4)
    })
    
    # Extract Feature Importances / Coefficients / Permutation Importances
    fitted_model = pipeline.named_steps['regressor']
    if hasattr(fitted_model, 'feature_importances_'):
        importances = fitted_model.feature_importances_
    elif hasattr(fitted_model, 'coef_'):
        importances = np.abs(fitted_model.coef_)
    else:
        # Fallback to Permutation Importance for models like KNN
        perm_imp = permutation_importance(pipeline, X_test, y_test, n_repeats=5, random_state=42)
        importances = perm_imp.importances_mean
        
    feature_importances_dict[name] = pd.Series(importances, index=X.columns)

# Metrics Summary DataFrame
results_df = pd.DataFrame(results).sort_values(by="RMSE", ascending=True)

print("=== Model Performance Comparison ===")
print(results_df.to_string(index=False))

# ---------------------------------------------------------
# 4. VISUALIZATION
# ---------------------------------------------------------
fig = plt.figure(figsize=(18, 12))

# Subplot 1: Model Comparison (R2 Score & RMSE)
ax1 = plt.subplot(2, 2, 1)
sns.barplot(data=results_df, x="R2 Score", y="Model", palette="viridis", ax=ax1)
ax1.set_title("Model Comparison - R² Score (Higher is Better)", fontsize=13, fontweight='bold')
ax1.set_xlim(0, 1.0)

ax2 = plt.subplot(2, 2, 2)
sns.barplot(data=results_df, x="RMSE", y="Model", palette="magma", ax=ax2)
ax2.set_title("Model Comparison - Root Mean Squared Error (Lower is Better)", fontsize=13, fontweight='bold')

# Subplot 3: Feature Importances (Tree Ensemble Baseline - Gradient Boosting)
ax3 = plt.subplot(2, 2, 3)
gb_imp = feature_importances_dict["Gradient Boosting"].sort_values(ascending=True)
gb_imp.plot(kind='barh', color='teal', ax=ax3)
ax3.set_title("Gradient Boosting Feature Importances", fontsize=13, fontweight='bold')
ax3.set_xlabel("Normalized Importance")

# Subplot 4: Feature Importance Comparison Across Tree & Linear Models
ax4 = plt.subplot(2, 2, 4)
importance_df = pd.DataFrame(feature_importances_dict)
# Normalize feature importances per model for cross-comparison plot
importance_df_norm = importance_df.apply(lambda x: x / (x.max() + 1e-9))
sns.heatmap(importance_df_norm, annot=True, fmt=".2f", cmap="Blues", ax=ax4, cbar=True)
ax4.set_title("Relative Feature Importance Matrix across Models", fontsize=13, fontweight='bold')

plt.tight_layout()
plt.show()